# Figure 3

**Linear permeability homogenization**

This notebook computes the numerically homogenized permeability of undeformed unit cells with round and hexagonal pores over a range of porosities, and compares the results with dilute and differential estimates.

---

**Authors**

Haotian XIAO
Martin GENET

École Polytechnique, Palaiseau, France


## Imports

In [ ]:
from pathlib import Path
import math

import dolfin
import myPythonLibrary as mypy
import dolfin_mech as dmech

## Output directory

In [ ]:
# All results and figures of this notebook are written to the subdirectory Fig3/
# of the notebook directory (demos/).
RESULTS_DIR = Path("Fig3").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR =", RESULTS_DIR)

## Homogenization function


In [ ]:
def run_linear_homogenization_hollowbox(
    r0_list,
    res_folder,
    shape_list=("round", "hex"),
    probe_list=("gx", "gy"),
    pressure_gradient_amplitude=0.01,
    perform_tests=0,
):
    res_folder = Path(res_folder)
    res_folder.mkdir(parents=True, exist_ok=True)

    test = mypy.Test(
        res_folder=str(res_folder),
        perform_tests=perform_tests,
        stop_at_failure=1,
        clean_after_tests=0,
        tester_numpy_tolerance=1e-2,
    )

    material_parameters = {
        "alpha": 0.16,
        "gamma": 0.5,
        "c1": 0.2,
        "c2": 0.4,
        "kappa": 1e2,
        "eta": 1e-5,
    }

    dim = 2
    n_steps = 1

    def run_one_probe(shape, r0, probe):
        if probe == "gx":
            gx_ini_lst = [0.0]
            gx_fin_lst = [pressure_gradient_amplitude]
            gy_ini_lst = [0.0]
            gy_fin_lst = [0.0]
        elif probe == "gy":
            gx_ini_lst = [0.0]
            gx_fin_lst = [0.0]
            gy_ini_lst = [0.0]
            gy_fin_lst = [pressure_gradient_amplitude]
        else:
            raise ValueError("probe must be 'gx' or 'gy'")

        res_basename = f"linear-{shape}-r0={r0}-{probe}"
        result_basename = res_folder / res_basename
        mesh_filebasename = res_folder / f"mesh-{shape}-r0={r0}"

        load_params = {
            "solid": {},
            "liquid": {},
            "air": {},
        }

        for i in range(dim):
            for j in range(dim):
                load_params["solid"][f"sigma_bar_{i}{j}"] = 0.0

        load_params["solid"]["U_bar_00_lst"] = [0.0]
        load_params["solid"]["U_bar_01_lst"] = [0.0]
        load_params["solid"]["U_bar_10_lst"] = [0.0]
        load_params["solid"]["U_bar_11_lst"] = [0.0]

        load_params["liquid"]["pl_bar_ini_lst"] = [0.0] * n_steps
        load_params["liquid"]["pl_bar_fin_lst"] = [0.0] * n_steps

        load_params["liquid"]["grad_p_bar_x_ini_lst"] = gx_ini_lst
        load_params["liquid"]["grad_p_bar_x_fin_lst"] = gx_fin_lst
        load_params["liquid"]["grad_p_bar_y_ini_lst"] = gy_ini_lst
        load_params["liquid"]["grad_p_bar_y_fin_lst"] = gy_fin_lst

        load_params["liquid"]["Theta_in_ini_lst"] = [0.0] * n_steps
        load_params["liquid"]["Theta_in_fin_lst"] = [0.0] * n_steps
        load_params["liquid"]["Theta_out_ini_lst"] = [0.0] * n_steps
        load_params["liquid"]["Theta_out_fin_lst"] = [0.0] * n_steps

        load_params["air"]["pf"] = 0.0
        load_params["air"]["pf_lst"] = [0.0] * n_steps

        dmech.run_HollowBox_MicroPoroflow(
            dim=dim,
            mesh_params={
                "dim": dim,
                "xmin": 0.0,
                "ymin": -math.sqrt(3.0) / 2.0,
                "xmax": 1.0,
                "ymax": math.sqrt(3.0) / 2.0,
                "r0": r0,
                "l": 0.03,
                "hole_shape": shape,
                "add_center_hole": True,
                "mesh_filebasename": str(mesh_filebasename),
            },
            mat_params={
                "skel": {
                    "parameters": material_parameters,
                    "scaling": "no",
                },
                "bulk": {
                    "parameters": material_parameters,
                    "scaling": "no",
                },
                "pore": {
                    "parameters": material_parameters,
                    "scaling": "no",
                },
            },
            flow_params={
                "k_l": dolfin.Constant(
                    (
                        (1.0, 0.0),
                        (0.0, 1.0),
                    )
                ),
                "use_kozeny_carman": False,
            },
            porosity_params={
                "type": "constant",
                "val": 0.3,
            },
            bcs="pbc",
            step_params={
                "n_steps": n_steps,
                "Deltat_lst": [1e-2],
                "dt_ini_lst": [1e-3],
                "dt_min_lst": [1e-4],
                "dt_max_lst": [5e-3],
            },
            load_params=load_params,
            res_basename=str(result_basename),
            verbose=0,
        )

        test.test(str(result_basename))

    for shape in shape_list:
        for r0 in r0_list:
            for probe in probe_list:
                print(f"Running shape={shape}, r0={r0}, probe={probe}")
                run_one_probe(
                    shape=shape,
                    r0=r0,
                    probe=probe,
                )

    return res_folder

## Parameters

In [ ]:
r0_list = [0.02, 0.05, 0.072, 0.1, 0.2, 0.315, 0.41, 0.45]
shape_list = ("round", "hex")
probe_list = ("gx", "gy")

res_folder = RESULTS_DIR / "Figure3_linear_homogenization"

## Computing homogenized permeability

In [ ]:
run_linear_homogenization_hollowbox(
    r0_list=r0_list,
    res_folder=res_folder,
    shape_list=shape_list,
    probe_list=probe_list,
)

## Plot Figure 3


In [ ]:
from Plot_Microporoflow import plot_linear_homogenization_hollowbox

figures_folder = RESULTS_DIR / "figures"
figures_folder.mkdir(parents=True, exist_ok=True)

figure_path = figures_folder / "Figure3_linear_homogenization_K_vs_phi.pdf"

plot_linear_homogenization_hollowbox(
    res_folder=str(res_folder),
    shape_list=shape_list,
    r0_list=r0_list,
    geometry_r0_values={"round": (0.072, 0.315, 0.41), "hex": (0.072, 0.315, 0.45)},
    save_name=str(figure_path),
    show_plot=True,
)

print("Figure saved in:", figure_path)

## Comparison with the Hashin–Shtrikman bound (Sec. 3.1)

In [ ]:
import Post_Microporoflow as pm

# Keq = (K1 + K2)/2 of the unit cells computed above against h(phi) = (1 - phi)/(1 + phi)
hs_table = pm.hs_check_linear(
    res_folder=res_folder,
    shape_list=shape_list,
    r0_list=r0_list,
    out_csv=figures_folder / "Figure3_HS_check.csv",
)
pm.print_hs_check(hs_table)